# <b>Agent Platform AutoML による物体検出</b>


## <b>学習目標</b> ##

1. Agent Platform（Vertex AI）上で物体検出用の管理対象データセット（Managed Dataset）を作成する方法を学ぶ
1. AutoML 物体検出モデルをトレーニングする方法を学ぶ
1. AutoML でトレーニングされたモデルを評価する方法を学ぶ
1. AutoML でトレーニングされたモデルをエンドポイントにデプロイする方法を学ぶ
1. デプロイされたモデルを使用して新しいデータに対して予測を行う方法を学ぶ


このノートブックでは、Agent Platform を使用して、指定された画像内の複数の物体を検出し、その物体の種類と画像内の位置に関する情報を提供する AutoML モデルをトレーニングします。

まず、Agent Platform 上で管理対象データセット（Managed Dataset）を作成し、一般公開されている画像セットをそこにインポートします。その後、AutoML モデルのトレーニング、評価、エンドポイントへのデプロイを行います。最後に、モデルに予測リクエストを送信します。


### セットアップ


In [ ]:
import os
from datetime import datetime

from google.cloud import aiplatform

PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT  # デフォルトは PROJECT
REGION = "us-central1"
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

os.environ["BUCKET"] = BUCKET
os.environ["PROJECT"] = PROJECT
os.environ["REGION"] = REGION

aiplatform.init(project=PROJECT, location=REGION)

## ステップ 1: 訓練データの準備とフォーマット

Agent Platform で管理対象データセットを作成する最初のステップは、訓練データを準備することです。このケースでは、訓練データセットは画像と、画像内の物体の位置（バウンディングボックスの座標）および種類（ラベル）を特定する情報で構成されます。
物体検出用の管理対象データセットを準備するための一般的な制約とルールは以下のとおりです:

* サポートされている画像形式: JPEG、PNG、GIF、BMP、ICO。1 画像あたりの最大ファイルサイズは 30MB です。

* ラベル（画像内で検出したい物体の種類）あたり約 1,000 枚の訓練用画像を用意することが推奨されます。各ラベルには少なくとも 10 枚の画像が必要で、それぞれに少なくとも 1 つのアノテーション（バウンディングボックスとラベル）が含まれている必要があります。一般に、ラベルあたりの画像数が多いほど、モデルのパフォーマンスは向上します。


### 画像 URI とラベルを含む CSV ファイルの作成

画像ファイルを Cloud Storage バケットにアップロードしたら、画像 URI、バウンディングボックス情報、および物体ラベルをリストした CSV（または JSONL）ファイルを作成する必要があります。CSV ファイルには画像内のバウンディングボックスごとに 1 行が含まれるため、2 つのバウンディングボックスを持つ画像の場合、同じ画像 URI を共有する 2 行が CSV ファイル内に存在することになります。

以下の例では、1 行目と 2 行目は 2 つのアノテーション
`(car,0.1,0.1,,,0.3,0.3,,)` および `(bike,.7,.6,,,.8,.9,,)` を持つ同じ画像を参照しています。アノテーションの最初の要素はバウンディングボックス内の物体ラベルであり、残りは画像内のバウンディングボックスの座標です（詳細は下記参照）。


3 行目はアノテーションが 1 つだけある画像 `(car,0.1,0.1,0.2,0.1,0.2,0.3,0.1,0.3)` を参照し、4 行目はアノテーションのない画像を参照しています。

1 列目はデータの分割に対応し（これは省略可能です。指定しない場合、AutoML は 80/10/10 の分割を行います）、2 列目は画像 URI、最後の列がアノテーションを保持します。

バウンディングボックスは、次の 2 つの形式のいずれかで指定できます:
- 2 つの頂点（長方形の対角にある 2 点の x, y 座標）を含む形式: `(x_min, y_min, , ,x_max,y_max,,)`、または
- 4 つの頂点すべてを含む形式: `(x_min,y_min,x_max,y_min,x_max,y_max,x_min,y_max)`
**例:**


```bash
TRAIN,gs://folder/image1.png,car,0.1,0.1,,,0.3,0.3,,
TRAIN,gs://folder/image1.png,bike,.7,.6,,,.8,.9,,
UNASSIGNED,gs://folder/im2.png,car,0.1,0.1,0.2,0.1,0.2,0.3,0.1,0.3
TEST,gs://folder/im3.png,,,,,,,,,
```

上記の各行には以下の列が含まれます:

1. <b>その行のコンテンツがどのデータセットに割り当てられているか。</b> - `TRAIN`、`VALIDATE`、`TEST`、または `UNASSIGNED`
1. <b>どのアノテーション対象コンテンツか。</b> - 画像の GCS URI が含まれます
1. <b>物体がどのように分類されているかを識別するラベル。</b>
1. <b>画像内の物体に対するバウンディングボックス。</b>


このラボでは、あらかじめ作成されている CSV を使用します。CSV 内の画像 URI は GCS 上で一般公開されている画像を指しており、各画像内の食品 / 食材に関連するアノテーションが含まれています。

GCS バケットが存在することを確認し（存在しない場合は作成）、CSV を GCS バケットにコピーします。


In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
   echo -e "Bucket gs://${BUCKET} already exists."
    
else
   echo "Creating a new GCS bucket."
   gcloud storage buckets create --location=${REGION} gs://${BUCKET}
   echo -e "\nHere are your current buckets:"
   gcloud storage ls
fi

CSV ファイルの先頭数行を確認し、GCS バケットにコピーします


In [ ]:
!head ../dataset_import_files/salads.csv

In [ ]:
!gcloud storage cp ../dataset_import_files/salads.csv gs://{BUCKET}/

## ステップ 2: 管理対象データセット（Managed Dataset）の作成

次のステップでは、Agent Platform 上で物体検出用の管理対象データセットを作成します。
<br/>
**注**: これには 5 〜 10 分ほどかかる場合があります。


In [ ]:
DATASET_NAME = "salad_dataset"

ds = aiplatform.ImageDataset.create(
    display_name=DATASET_NAME,
    gcs_source=f"gs://{BUCKET}/salads.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.bounding_box,
)

## ステップ 3: AutoML モデルのトレーニング
トレーニングジョブを開始します。`job.run()` に以下のパラメータを設定します:
- `dataset`: 先ほど作成した管理対象データセット
- `model_display_name`: モデルの表示名。トレーニング完了後、トレーニング済みモデルは Model オブジェクトになります。
- `training_fraction_split`: 訓練に使用するデータの割合。明示的に 0.8（80%）に設定します。
- `validation_fraction_split`: 検証に使用するデータの割合。明示的に 0.1（10%）に設定します。
- `test_fraction_split`: テストに使用するデータの割合。明示的に 0.1（10%）に設定します。
- `budget_milli_node_hours`: トレーニングジョブのミリノード時間（例: 1000 は 1 ノード時間）。AutoML 物体検出トレーニングジョブの最小値は 20000（20 ノード時間）です。
- `disable_early_stopping`: ブール値。明示的に False に設定します（つまり早期終了を有効にします）。


In [ ]:
MODEL_DISPLAY_NAME = "salad"

job = aiplatform.AutoMLImageTrainingJob(
    display_name="salad_" + TIMESTAMP, prediction_type="object_detection"
)

model = job.run(
    dataset=ds,
    model_display_name=MODEL_DISPLAY_NAME,
    training_fraction_split=0.8,
    validation_fraction_split=0.1,
    test_fraction_split=0.1,
    budget_milli_node_hours=20000,
    disable_early_stopping=False,
)

**注:** モデルのトレーニングが完了するまでに 1.5 〜 2.5 時間かかる場合があります。先に進む前に、モデルのトレーニングが完了するまで待つ必要があります。

モデルリソースを取得し、評価指標を確認します。ここでは [Mean Average Precision](https://blog.paperspace.com/mean-average-precision/)（mAP: 平均適合率の平均）を確認します。物体検出モデルの評価指標についての詳細は、[こちら](https://pyimagesearch.com/2016/11/07/intersection-over-union-iou-for-object-detection/) を参照してください。


In [ ]:
# 表示名でモデルリソースを取得
models = aiplatform.Model.list(filter="display_name=salad")
salad_model = models[0]

# モデルの評価を取得
client_options = {"api_endpoint": f"{REGION}-aiplatform.googleapis.com"}
model_service_client = aiplatform.gapic.ModelServiceClient(
    client_options=client_options
)

model_evaluations = model_service_client.list_model_evaluations(
    parent=salad_model.resource_name
)
evals = list(model_evaluations)[0]

for metric in list(evals.metrics["boundingBoxMetrics"]):
    try:
        print(f"Mean Average Precision (mAP): {metric['meanAveragePrecision']}")
    except KeyError:
        print("IOU threshold is too high to detect positive cases.")

    print(f"Intersection over union (IoU): {metric['iouThreshold']}\n")

## ステップ 6: モデルのデプロイ

トレーニングされたモデルのパフォーマンスに満足したら、API 経由で予測を利用できるようにデプロイします。


In [ ]:
ENDPOINT_DISPLAY_NAME = "salad_model_endpoint"

endpoint = salad_model.deploy(
    deployed_model_display_name=ENDPOINT_DISPLAY_NAME,
    traffic_split=None,
    min_replica_count=1,
    max_replica_count=1,
)

## ステップ 7: 予測リクエストの送信

この例では、プロジェクトの Cloud Storage バケットに保存されている画像から個別の予測を実行します。
物体検出モデルは、入力画像に対して多数のバウンディングボックスを出力します。出力として、各ボックスに以下が付随することを想定しています:
1. ラベル
1. 信頼度スコア（confidence score）

予測を取得するためのデータ準備には、以下が含まれます:
1. `base64.b64encode` で画像をエンコードし（エンコードされたバイト列が返されます）、`.decode("utf-8")` を適用してデコードされた文字列を取得する。
1. `ImageObjectDetectionPredictionInstance` を作成する。
1. `ImageObjectDetectionPredictionParams` を作成する。これは主に信頼度のしきい値を設定するために使用されます。
1. 予測リクエストを実行する。


In [ ]:
import base64

from google.cloud.aiplatform.gapic.schema import predict

TEST_IMAGE = "../test_images/salad.jpg"  # Cloud Storage バケットにアップロードした任意の画像に置き換えてください

# ファイルを読み込む。
with open(TEST_IMAGE, "rb") as f:
    file_content = f.read()

encoded_content = base64.b64encode(file_content).decode("utf-8")
instance = predict.instance.ImageObjectDetectionPredictionInstance(
    content=encoded_content
).to_value()

parameters = predict.params.ImageObjectDetectionPredictionParams(
    confidence_threshold=0.7
).to_value()

response = endpoint.predict(instances=[instance], parameters=parameters)
preds = response.predictions[0]
preds

デプロイされたモデルからレスポンスオブジェクトを取得できたので、その予測結果（つまり、上のセルでモデルに送信した画像からモデルが検出したバウンディングボックスと物体）を確認できます:

`preds` は以下のキーを持つ Python 辞書です:
* `bboxes`: 予測されたバウンディングボックスの座標リスト
* `displayNames`: バウンディングボックスの予測ラベルのリスト
* `confidences`: 予測信頼度のリスト


In [ ]:
from itertools import cycle

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

# 各ラベルに異なる色を生成するため
cycol = cycle("bgrcmk")

img = Image.open(TEST_IMAGE)
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(preds["displayNames"])):
    bbox = preds["bboxes"][i]
    display_name = preds["displayNames"][i]
    confidence = preds["confidences"][i]
    xy = (int(bbox[0] * img.width), int(bbox[2] * img.height))
    w = int(img.width * (bbox[1] - bbox[0]))
    h = int(img.height * (bbox[3] - bbox[2]))
    r = patches.Rectangle(
        xy,
        w,
        h,
        linewidth=1,
        facecolor="none",
        label=f"{display_name}: {confidence:.3f}",
        edgecolor=next(cycol),
    )
    ax.add_patch(r)

plt.legend()
plt.show()

お疲れ様でした！このラボでは、物体検出用の Agent Platform 管理対象データセットを作成しました。その後、AutoML モデルをトレーニングし、トレーニング済みモデルをエンドポイントにデプロイして予測を提供できるようにしました。
